# Additional processing of human just-think data

In [ ]:
import os
import json
import random
from ast import literal_eval

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

import analysis_utils
from analysis_utils import compute_utility, get_pwin
import models.utils as utils
from constants import THINK_HUMAN_DATA, THINK_STIMULI_PTH

%matplotlib inline

OUT_DIR = "final_processed_res/think"
FIG_DIR = "final_figures"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# Load human data + game index
_, idx2game, game2idx, _ = analysis_utils.process_game_stimuli(THINK_STIMULI_PTH)
human_df = pd.read_csv(THINK_HUMAN_DATA)

ordered_games = sorted(set(human_df.game_id))
tasks = ["advantage", "how_fun"]


## Scratchpad usage

In [ ]:
# Per-user scratchpad rollout counts, by task
task2user = {task: set() for task in tasks}
user_scratchpad_usage = {}

for _, entry in human_df.iterrows():
    task = entry.task
    user_ids = literal_eval(entry.user_ids)
    scratch_uses = literal_eval(entry.scratchpad_rollouts)
    for uid, n_rollouts in zip(user_ids, scratch_uses):
        user_scratchpad_usage.setdefault(uid, []).append(n_rollouts)
        task2user[task].add(uid)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
bars = np.arange(0, 7)
for ax, task in zip(axes, tasks):
    usage = np.array(
        sum([v for uid, v in user_scratchpad_usage.items() if uid in task2user[task]], [])
    )
    counts = [(usage == val).sum() for val in bars]
    ax.bar(bars, counts, width=0.9, align="center", alpha=0.6,
           label=task.split("how_")[-1].capitalize())
    ax.set_xticks(bars)
    ax.set_xticklabels([str(x) for x in bars])
    ax.set_xlabel("Num Rollouts", fontsize=18)
    ax.set_ylabel("Count", fontsize=18)
    ax.legend(fontsize=16)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/agg_scratchpad.pdf", dpi=400)
plt.show()

## Per-game human payoff judgments

In [ ]:
# Per game payoff judgements, from which we can compute various quantities
human_payoff = {}
task_df = human_df[human_df.task == "advantage"]

for game in ordered_games:
    game_row = task_df[task_df.game_id == game].iloc[0]
    payoffs, ents, emds, pdraws, pwins = [], [], [], [], []
    for resp in literal_eval(game_row.all_scores):
        pdraw = resp["draw_response"] / 100
        pwin = get_pwin(resp["draw_response"], resp["firstplayer_response"]) / 100
        plose = 1 - (pwin + pdraw)
        outcome_dist = {"lose": plose, "win": pwin, "draw": pdraw}

        payoffs.append(compute_utility(pdraw, pwin))
        ents.append(utils.compute_entropy([plose, pwin, pdraw]))
        emds.append(utils.compute_emd(outcome_dist))
        pdraws.append(pdraw)
        pwins.append(pwin)
    human_payoff[game] = {
        "payoff": payoffs, "ent": ents, "emd": emds,
        "pdraw": pdraws, "pwin": pwins,
    }

## Per-game raw funness scores

In [ ]:
tictactoe_id = "3.0*3.0*3 pieces in a row wins."
fun_df = human_df[human_df.task == "how_fun"]

uid2tictactoe = {}
uid2all_scores = {}
ttt_row = fun_df[fun_df.game_id == tictactoe_id].iloc[0]
for score, uid in zip(literal_eval(ttt_row.all_scores), literal_eval(ttt_row.user_ids)):
    uid2tictactoe[uid] = score
    uid2all_scores.setdefault(uid, []).append(score)
for game in ordered_games:
    if game == tictactoe_id:
        continue
    row = fun_df[fun_df.game_id == game].iloc[0]
    for score, uid in zip(literal_eval(row.all_scores), literal_eval(row.user_ids)):
        if uid in uid2all_scores:
            uid2all_scores[uid].append(score)

human_fun = {}
for game in ordered_games:
    row = fun_df[fun_df.game_id == game].iloc[0]
    scores = []
    for score, uid in zip(literal_eval(row.all_scores), literal_eval(row.user_ids)):
        if uid not in uid2tictactoe:
            continue
        scores.append(score)
    human_fun[game] = scores

## Save

In [ ]:
out_pth = f"{OUT_DIR}/human_processed.json"
with open(out_pth, "w") as f:
    json.dump({"human_payoff": human_payoff, "human_fun": human_fun}, f)
print(f"Wrote {out_pth}: {len(human_payoff)} games (payoff), {len(human_fun)} games (fun)")